### Generating Product metadata set for semantic search 

In [0]:
# Product Metadata Dataset (Aligned with Sales Data)
# This dataset is aligned with transactional data to enable
# semantic search and linkage with analytics (Gold layer).

products = [
    {
        "product_id": 1,
        "product_name": "Laptop",
        "category": "Electronics",
        "description": "High performance laptop suitable for work, gaming, and productivity tasks"
    },
    {
        "product_id": 2,
        "product_name": "Phone",
        "category": "Electronics",
        "description": "Smartphone with advanced camera features, long battery life, and fast performance"
    },
    {
        "product_id": 3,
        "product_name": "Tablet",
        "category": "Electronics",
        "description": "Portable tablet ideal for reading, browsing, and entertainment on the go"
    },
    {
        "product_id": 4,
        "product_name": "Headphones",
        "category": "Electronics",
        "description": "Wireless headphones with noise cancellation and immersive sound quality"
    }
]

# Quick validation
for p in products:
    print(p["product_name"], "->", p["description"])

Laptop -> High performance laptop suitable for work, gaming, and productivity tasks
Phone -> Smartphone with advanced camera features, long battery life, and fast performance
Tablet -> Portable tablet ideal for reading, browsing, and entertainment on the go
Headphones -> Wireless headphones with noise cancellation and immersive sound quality


#### This dataset is aligned with transactional product data, enabling semantic search results to be linked with sales analytics.

### Generate embeddings for product descriptions

In [0]:
# Convert descriptions into vector embeddings

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [p["description"] for p in products]
embeddings = model.encode(texts)

for i, p in enumerate(products):
    p["embedding"] = embeddings[i]

print(products[0]["product_name"], "embedding added")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Laptop embedding added


### Store embeddings in vector database

In [0]:
# Store embeddings with metadata (simulating vector DB)

vector_db = []

for p in products:
    vector_db.append({
        "product_id": p["product_id"],
        "product_name": p["product_name"],
        "category": p["category"],
        "description": p["description"],
        "embedding": p["embedding"]
    })

print(vector_db[0])

{'product_id': 1, 'product_name': 'Laptop', 'category': 'Electronics', 'description': 'High performance laptop suitable for work, gaming, and productivity tasks', 'embedding': array([ 1.8200150e-02,  5.4026932e-02, -2.6981235e-02,  1.1733744e-02,
       -1.9509556e-02,  8.6844265e-03,  1.3066540e-02, -7.8110206e-03,
       -9.7171813e-03,  4.2729612e-02, -1.4314060e-02,  7.2841078e-02,
       -2.7030045e-02,  2.5967799e-02,  8.3579481e-02,  3.0709162e-02,
        1.2741396e-01, -6.8892971e-02,  5.2804824e-02, -1.0992079e-01,
        2.8127920e-02,  1.8877255e-02,  1.0560813e-02, -1.9247063e-02,
        3.2929543e-02,  3.3557002e-02,  2.0635223e-02,  2.3502756e-02,
       -7.7814400e-02, -1.3018771e-02, -1.1644130e-02, -7.7617392e-03,
       -1.9963548e-02,  5.4241203e-02, -1.0263394e-02,  2.7325572e-02,
        3.0597422e-02, -8.3673331e-06,  7.3536776e-02, -7.5495049e-02,
       -7.0184886e-02, -2.4917373e-02,  5.4136779e-02, -8.1503261e-03,
        6.2644638e-02, -1.4995806e-02, -3.6

### Implement semantic search

In [0]:
# Perform semantic search using cosine similarity

from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query):
    query_vec = model.encode([query])
    
    results = []
    
    for item in vector_db:
        sim = cosine_similarity([item["embedding"]], query_vec)[0][0]
        results.append((item["product_name"], sim))
    
    return sorted(results, key=lambda x: x[1], reverse=True)[:2]

### Validate semantic search results

In [0]:
# Display semantic search results in a clean and readable format

from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query):
    query_vec = model.encode([query])
    
    results = []
    for item in vector_db:
        sim = cosine_similarity([item["embedding"]], query_vec)[0][0]
        results.append((item["product_name"], float(sim)))
    
    return sorted(results, key=lambda x: x[1], reverse=True)[:2]


def display_results(query):
    results = semantic_search(query)
    
    print(f"\nQuery: {query}")
    for product, score in results:
        print(f"Product: {product} | Similarity: {round(score, 3)}")


# Test queries
display_results("gaming laptop")
display_results("camera smartphone")
display_results("noise cancelling audio")


Query: gaming laptop
Product: Laptop | Similarity: 0.714
Product: Tablet | Similarity: 0.371

Query: camera smartphone
Product: Phone | Similarity: 0.808
Product: Tablet | Similarity: 0.257

Query: noise cancelling audio
Product: Headphones | Similarity: 0.644
Product: Laptop | Similarity: 0.004


#### This implementation demonstrates semantic search where user queries are matched to products based on meaningrather than exact keyword matching.

#### Example:
-  "gaming laptop" → matches Laptop
-  "noise cancelling audio" → matches Headphones

This enhances product discovery beyond traditional search.

#### 🚀 Business value
##### Better product discovery
- Users find relevant products even with vague queries
- Less dependency on exact keywords
##### Higher conversion rate
- Better search → more clicks → more purchases
##### Reduced search friction
- Users don’t need to “guess” correct product names
##### Supports intelligent recommendations
- Can suggest similar products based on meaning